In [2]:
import os
import librosa
import numpy as np
import soundfile as sf
from tqdm import tqdm

# THE FIX: Add "../" to tell Python to go up one folder level first
INPUT_DIR = "../data/audio/"               # Points to MINDTRACK_AI/data/audio/
OUTPUT_DIR = "../data/audio_processed/"    # Creates a new folder: MINDTRACK_AI/data/audio_processed/

all_files = []
for root, _, files in os.walk(INPUT_DIR):
    for f in files:
        if f.endswith((".wav", ".mp3")):
            all_files.append(os.path.join(root, f))

# Check if it found them before running the heavy processing!
print(f"Found {len(all_files)} audio files to process.")

# (The rest of your processing loop stays exactly the same)
for input_path in tqdm(all_files, desc="Processing Audio Data"):
    try:
        y, sr = librosa.load(input_path, sr=22050, mono=True)
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        
        rms = np.sqrt(np.mean(y_trimmed**2))
        y_final = y_trimmed / rms if rms > 0 else y_trimmed
        
        file_name = os.path.basename(input_path)
        relative_path = os.path.relpath(os.path.dirname(input_path), INPUT_DIR)
        output_folder = os.path.join(OUTPUT_DIR, relative_path)
        os.makedirs(output_folder, exist_ok=True)
        
        sf.write(os.path.join(output_folder, file_name), y_final, sr)

    except Exception as e:
        tqdm.write(f"Failed on {input_path}: {e}")

Found 2452 audio files to process.


Processing Audio Data: 100%|██████████| 2452/2452 [01:04<00:00, 38.04it/s]


In [3]:
from pathlib import Path
import numpy as np
import librosa

OUTPUT_DIR = "../data/audio_processed/"
processed = list(Path(OUTPUT_DIR).rglob("*.wav"))
print(f"Total processed files : {len(processed)}")  # should match your input count

# Spot-check one file
y, sr = librosa.load(processed[0], sr=None)
rms = np.sqrt(np.mean(y**2))
print(f"Sample rate  : {sr}")
print(f"Duration     : {len(y)/sr:.2f}s")
print(f"RMS level    : {rms:.4f}")  # should be close to 0.1

Total processed files : 2452
Sample rate  : 22050
Duration     : 1.23s
RMS level    : 0.6169


In [2]:
import pandas as pd
# Create a small summary table of your processed audio
audio_stats = {
    "Metric": ["Total Files", "Sample Rate", "Duration (Avg)", "RMS Level (Normalized)"],
    "Value": ["2452", "22050 Hz", "1.23s", "0.6169"]
}
display(pd.DataFrame(audio_stats))

,Metric,Value
0,Total Files,2452
1,Sample Rate,22050 Hz
2,Duration (Avg),1.23s
3,RMS Level (Normalized),0.6169
